In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
from ariel_pred.dataset import DataLoaderAndCalibrator
from ariel_pred.preprocessing import SGSmoothing
from ariel_pred.plots import plot_white_curve
from ariel_pred.transit import FunctionFittingBasedPhaseDetector
from ariel_pred.models import TransitMultiplicationFactorFinder, SValuesCNN, SValuesCNNTrainer
from ariel_pred.features import WavelengthsGroupsMultiplierFinder
from ariel_pred.metrics import score, gll
from ariel_pred.sigma import OGSignalVarBasedSigmaCalculator
from ariel_pred.modeling.s_values_cnn_with_star_info import SValuesCNNWithStarInfoModel, SValuesCNNWithStarInfoTrainer
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.optimize import minimize
from tqdm.auto import tqdm
import pandas as pd
import torch
from torch import nn
import scipy
from ariel_pred.sigma import LROnVariousFeaturesSigmaCalculator

In [3]:
data_loader = DataLoaderAndCalibrator(
    data_path=Path("../data/raw"),
    output_path=Path("../data/calibrated/full"),
    force_recalibration=False,
    cut_airs_channels=True,
    binning=4,
    n_jobs=4
)
train_data, train_labels = data_loader.load_all_train_data()
train_data.shape, train_labels.shape

Loading calibrated train data...


((1100, 1406, 283), (1100, 283))

In [4]:
features_extractor = WavelengthsGroupsMultiplierFinder()

features = features_extractor.extract_features(
    train_data,
    average_cross_groups=False,
    wavelengths_groups=[1, 2, 4, 8, 16, 32, 64],
    weights=[1, 1, 1, 1, 1, 1, 1]
)

features.shape

  0%|          | 0/1100 [00:00<?, ?it/s]

(1100, 283, 7)

In [5]:
model_data = features.transpose(0, 2,1)
model_data.shape

(1100, 7, 283)

In [6]:
star_info_df = pd.read_csv("../data/raw/train_star_info.csv")
star_info_df.head()

,planet_id,Rs,Ms,Ts,Mp,e,P,sma,i
0,34983,1.155435,1.062961,5577.006645,0.694946,0.0,3.305589,8.550786,89.150759
1,1873185,1.813230,1.370451,6216.229756,0.610845,0.0,6.352660,9.553384,88.701514
2,3849793,0.653807,0.667352,4968.477186,1.529200,0.0,5.522798,15.285661,89.134177
3,8456603,1.250623,1.162019,6023.702622,2.262107,0.0,7.541019,14.144310,87.178007
4,23615382,1.431492,1.306489,6128.061013,0.861299,0.0,4.368080,8.347654,90.000000


In [7]:
star_info = star_info_df[["Rs", 'i']].values
star_info.shape

(1100, 2)

In [9]:
trainer = SValuesCNNWithStarInfoTrainer(
    Path("../models/s_values_cnn_with_star_info_new"),
    n_splits=5,
    num_channels=model_data.shape[1],
    wavelengths=model_data.shape[2],
    num_star_features=star_info.shape[1]
)
val_losses, train_losses, val_preds_list = trainer.train(model_data*1e3, star_info, train_labels*1e3, epochs=300)

Fold 1/5


Training:   0%|          | 0/300 [00:00<?, ?it/s]

Loading weights from ../models/s_values_cnn_with_star_info_new/model_fold_1.pt
Fold 2/5


Training:   0%|          | 0/300 [00:00<?, ?it/s]

Loading weights from ../models/s_values_cnn_with_star_info_new/model_fold_2.pt
Fold 3/5


...

Loading weights from ../models/s_values_cnn_with_star_info_new/model_fold_3.pt
Fold 4/5


Training:   0%|          | 0/300 [00:00<?, ?it/s]

Loading weights from ../models/s_values_cnn_with_star_info_new/model_fold_4.pt
Fold 5/5


Training:   0%|          | 0/300 [00:00<?, ?it/s]

Loading weights from ../models/s_values_cnn_with_star_info_new/model_fold_5.pt


In [10]:
val_preds_list

array([[18.415812 , 18.43969  , 18.473007 , ..., 18.418095 , 18.45858  ,
        18.454285 ],
       [ 6.1284223,  6.095124 ,  6.0945888, ...,  6.157889 ,  6.1474447,
         6.143238 ],
       [46.553253 , 46.696896 , 46.664192 , ..., 46.783695 , 46.76163  ,
        46.689453 ],
       ...,
       [ 6.6078835,  6.6025157,  6.5993547, ...,  6.59178  ,  6.62337  ,
         6.6011043],
       [14.649672 , 14.769006 , 14.790927 , ..., 14.795151 , 14.796701 ,
        14.784613 ],
       [13.3626585, 13.310565 , 13.296089 , ..., 13.37763  , 13.401862 ,
        13.358406 ]], shape=(1100, 283), dtype=float32)

In [11]:
predictions = val_preds_list / 1e3
predictions.shape

(1100, 283)

In [21]:
np.save("../space_data/predictions.npy", predictions)

In [13]:
spectrum = predictions

In [15]:
sigma_calculator = LROnVariousFeaturesSigmaCalculator(
        mean_fgs_sigma=8.5e-4,
        mean_airs_sigma=6.5e-4,
        fgs_min_sigma=1e-6,
        airs_min_sigma=1e-6,
    )

val_sigma = sigma_calculator.train(data=train_data, spectrum=spectrum, labels=train_labels)

  0%|          | 0/1100 [00:00<?, ?it/s]

Calculating AIRS sigma:   0%|          | 0/1100 [00:00<?, ?it/s]

  0%|          | 0/1100 [00:00<?, ?it/s]

Fold 1, Validation RMSE: 0.0005725265722508685
Fold 2, Validation RMSE: 0.000273649646182893
Fold 3, Validation RMSE: 0.0004999397826155723
Fold 4, Validation RMSE: 0.00045310102292722937
Fold 5, Validation RMSE: 0.0005607982621268855


In [16]:
val_sigma.shape

(1100, 283)

In [20]:
np.save("../space_data/sigma.npy", val_sigma)

In [29]:
smoother = SGSmoothing(window_size=150, poly_order=2)
train_data

array([[[ 2240.80531207,  9205.66354332, 10268.06244144, ...,
           504.87124943,   492.79657974,   458.89989268],
        [ 2239.88073774,  9189.0767849 , 10252.66487151, ...,
           503.38478535,   491.94007762,   459.02243173],
        [ 2238.98864959,  9205.40099737, 10261.77984892, ...,
           505.06485904,   493.17841134,   460.98032301],
        ...,
        [ 2238.85054146,  9237.90552567, 10304.73516265, ...,
           500.04869698,   490.09079194,   460.20004768],
        [ 2240.18199407,  9241.43730845, 10303.80957348, ...,
           502.48412255,   490.6795692 ,   458.45572685],
        [ 2240.04282001,  9234.19547161, 10301.18960044, ...,
           506.62692143,   496.73731164,   463.39569736]],

       [[ 1821.42107233,  5159.83453365,  5757.89791417, ...,
           278.31765621,   272.59352922,   254.07336762],
        [ 1819.44984609,  5150.84942252,  5745.49411969, ...,
           277.6627771 ,   270.59763945,   250.32196983],
        [ 1820.51640599, 

In [27]:
train_data[0, :, 10:].shape

(1406, 273)

In [30]:
smoothed_signals = []
for signal in train_data:
    smoothed_signal = smoother.smooth(signal[:, 1:].mean(axis=1))
    smoothed_signals.append(smoothed_signal)
white_curves = np.array(smoothed_signals)
white_curves.shape

(1100, 1406)

In [31]:
np.save("../space_data/white_curves.npy", white_curves)

In [33]:
transit_finder = FunctionFittingBasedPhaseDetector()
phases = transit_finder.phase_detect_multiple_planets(white_curves)

  0%|          | 0/1100 [00:00<?, ?it/s]

In [34]:
phases.shape

(1100, 4)

In [35]:
np.save("../space_data/transit.npy", phases)

In [36]:
mean_out_of_transit = []

for curve, (t0, t1, t2, t3) in zip(white_curves, phases):
    mean_out_of_transit.append(np.mean(np.concatenate([curve[:t0], curve[t3:]])))

mean_out_of_transit = np.array(mean_out_of_transit)
mean_out_of_transit.shape

(1100,)

In [38]:
normalized_white_curves = white_curves / mean_out_of_transit[:, None]
normalized_white_curves.shape

(1100, 1406)

In [40]:
np.save("../space_data/normalized_white_curves.npy", normalized_white_curves)